In [ ]:
#sheets
import sys
from pathlib import Path

# sobe 1 nível (de notebooks → raiz do projeto)
ROOT = Path.cwd().parents[0]

sys.path.append(str(ROOT))

In [5]:
from src.config import PROJECT_ROOT, CREDENTIALS_PATH

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CREDENTIALS_PATH:", CREDENTIALS_PATH)

PROJECT_ROOT: C:\Users\Elcks\OneDrive\Documentos\Dataset Estudos\Chocolate Sales Kaggle
CREDENTIALS_PATH: C:\Users\Elcks\OneDrive\Documentos\Dataset Estudos\Chocolate Sales Kaggle\credentials\api_google_credentials.json


In [6]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Adiciona a pasta raiz do projeto no path
ROOT = Path.cwd().parents[0]  # notebooks/ -> projeto/
sys.path.append(str(ROOT))

from src.config import DATA_DIR
from src.io import load_data
from src.cleaning import normalize_columns, basic_null_report, clean_currency, clean_sales_df
from src.viz import plot_numeric_hist


In [ ]:
df = load_data(DATA_DIR / "chocolate_sales_2.csv")
df = clean_sales_df(df)
df.head()


In [8]:
from src.config import SHEET_ID, CREDENTIALS_PATH
from src.io import load_google_sheet

sheets = load_google_sheet(
    sheet_id=SHEET_ID,
    worksheet_name="Operações",
    credentials_path=CREDENTIALS_PATH
)

sheets.head()

,data,service_delivery_origin,service_delivery_next,service_delivery_final,total
0,01/01/2025,Belo Horizonte - MG,CD Duque de Caxias - RJ,Domicílio - Curitiba Sul,982.952
1,01/01/2025,Recife - PE,CD Camaçari - BA,Domicílio - Zona Leste SP,1.176.325
2,01/01/2025,Porto Alegre - RS,CD São José dos Pinhais - PR,Ponto de Retirada - Locker Shopping,959.893
3,02/01/2025,Rio de Janeiro - RJ,CD Barueri - SP,Ponto de Retirada - Locker Shopping,758.903
4,02/01/2025,Fortaleza - CE,CD São José dos Pinhais - PR,Domicílio - Zona Leste SP,1.369.429


In [ ]:
# KPIs principais
total_sales_value = df["amount"].sum().round(0)
total_countries = df["country"].nunique()
total_products = df["product"].nunique()
count_sales_persons = df["sales_person"].nunique()
count_sales = len(df)
total_sales_by_country = df.groupby("country")["amount"].sum().sort_values(ascending=False)

In [ ]:
# Relatório de valores nulos
basic_null_report(df)

In [ ]:
# Vendas por produto (top 15)
total_sales_by_product = df.groupby("product")["amount"].sum().sort_values(ascending=False).head(15)
ax = total_sales_by_product.plot(kind="barh", figsize=(10, 6))
ax.set_xlabel("Total de vendas ($)")
ax.set_title("Top 15 produtos por faturamento")
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 vendedores por faturamento
sales_by_person = df.groupby("sales_person")["amount"].sum().sort_values(ascending=False).head(10)
ax = sales_by_person.plot(kind="barh", figsize=(10, 5))
ax.set_xlabel("Total de vendas ($)")
ax.set_title("Top 10 vendedores por faturamento")
plt.tight_layout()
plt.show()

In [ ]:
# Série temporal: vendas ao longo do tempo (requer data em datetime)
df_date = df.copy()
df_date["date"] = pd.to_datetime(df_date["date"], format="%d/%m/%Y", errors="coerce")
monthly = df_date.dropna(subset=["date"]).groupby(df_date["date"].dt.to_period("M"))["amount"].sum()
ax = monthly.plot(kind="line", figsize=(12, 4), marker="o")
ax.set_xlabel("Mês")
ax.set_ylabel("Total de vendas ($)")
ax.set_title("Evolução das vendas por mês")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Histograma do valor das vendas (amount)
plot_numeric_hist(df, "amount", bins=40)